# Day 10: Human-in-the-Loop — Pause Before You Act

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Human approval is not a weakness. It is product design.

Some agent actions should never happen without a human saying "yes" — sending emails,
spending money, deleting data. Today we add approval gates before the agent acts.

## What You'll Build
- An agent that proposes an action and waits for approval
- The same pattern in all 3 frameworks
- See which framework makes human approval easiest


## Today's Scenario

The agent picks **one of your favourite dishes** — butter chicken, southern fried burger,
or shawarma — and proposes it. Before it serves up a recipe, **you must approve** the choice.

If approved → it gives the recipe. If rejected → nothing happens.

A simple "propose → human gate → execute" loop that works great even on a small local model.


In [2]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 10 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (9 model(s) available)

DAY 10 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


---
## Framework 1: OpenAI Agents SDK — Gate Between Two Runs

The SDK has no built-in "interrupt", so we place the human gate **between two `Runner.run()`
calls**. Step 1 proposes a dish; the cell blocks on `input()`; Step 2 gives the recipe only
if you approve.


In [3]:
from agents import Agent, Runner

model = get_openai_agents_model()

chef = Agent(
    name="Chef",
    instructions=(
        "You are a chef who picks ONE dish from: butter chicken, southern fried burger, shawarma. "
        "When asked to propose, reply with just the dish name. "
        "When asked for a recipe, give a short recipe."
    ),
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — DINNER HITL")
print("=" * 60)

# Step 1: agent proposes a dish
dish = (await Runner.run(chef, "Propose one dinner dish for me.")).final_output.strip()
print(f"\nAgent proposes: {dish}")

# ─── HITL GATE ─── (genuinely blocks until you type)
decision = input(f"\nApprove {dish}? (yes/no): ").strip().lower()

# Step 2: recipe only if approved
if decision == "yes":
    recipe = (await Runner.run(chef, f"Give a short recipe for {dish}.")).final_output
    print(f"\n--- RECIPE for {dish} ---")
    print(recipe)
else:
    print(f"\nREJECTED. No recipe for {dish}.")

print("\nNote: the gate lives BETWEEN two Runner.run() calls — propose,")
print("then execute only after the human says yes.")

OPENAI AGENTS SDK — DINNER HITL

Agent proposes: Butter Chicken

--- RECIPE for Butter Chicken ---
Butter Chicken Recipe:

Ingredients:
- 1 kg chicken thighs, cubed
- 2 tbsp vegetable oil
- 5 cloves garlic, minced
- 4 shallots, finely chopped
- 3-inch piece of ginger, finely grated
- 1 cup tomato puree
- 1 tsp ground cumin seeds
- 1 tsp ground coriander seeds
- 1 tsp turmeric powder
- 2 bay leaves
- 6绿cloves (green cardamoms)
- 1 cinnamon stick
- 1/3 cup garam masala powder
- Salt to taste
- 500 ml heavy cream
- 2 tbsp butter

Instructions:

1. In a large pan or skillet, heat oil over medium heat.
2. Add garlic, shallots, and ginger and sauté for about 2 minutes until they are fragrant.
3. Pour in the tomato puree and reduce the mixture by half, stirring continuously to prevent sticking.
4. Mix together cumin, coriander, turmeric powder, bay leaves, green cardamoms, cinnamon stick, garam masala, salt, and bring it to a boil. Once boiling, lower heat and cook for about 20 minutes until 

### Run 2 — reject at the gate

Same code as above; this time answer `no` at the prompt to watch the gate block the recipe.

In [4]:
from agents import Agent, Runner

model = get_openai_agents_model()

chef = Agent(
    name="Chef",
    instructions=(
        "You are a chef who picks ONE dish from: butter chicken, southern fried burger, shawarma. "
        "When asked to propose, reply with just the dish name. "
        "When asked for a recipe, give a short recipe."
    ),
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — DINNER HITL")
print("=" * 60)

# Step 1: agent proposes a dish
dish = (await Runner.run(chef, "Propose one dinner dish for me.")).final_output.strip()
print(f"\nAgent proposes: {dish}")

# ─── HITL GATE ─── (genuinely blocks until you type)
decision = input(f"\nApprove {dish}? (yes/no): ").strip().lower()

# Step 2: recipe only if approved
if decision == "yes":
    recipe = (await Runner.run(chef, f"Give a short recipe for {dish}.")).final_output
    print(f"\n--- RECIPE for {dish} ---")
    print(recipe)
else:
    print(f"\nREJECTED. No recipe for {dish}.")

print("\nNote: the gate lives BETWEEN two Runner.run() calls — propose,")
print("then execute only after the human says yes.")

OPENAI AGENTS SDK — DINNER HITL

Agent proposes: Butter Chicken

REJECTED. No recipe for Butter Chicken.

Note: the gate lives BETWEEN two Runner.run() calls — propose,
then execute only after the human says yes.


---
## Framework 2: LangGraph — `interrupt()` (the gold standard)

LangGraph's `interrupt()` makes the graph **freeze** at a node. Its state is checkpointed,
and it resumes only when you inject a value via `Command(resume=...)` — even from another
process or session.

> **Durability caveat:** this demo uses `MemorySaver`, which lives in-process and dies with
> the kernel. The *API* is identical for a real checkpointer — swap `MemorySaver()` for
> `SqliteSaver` or `PostgresSaver` in production, and the graph genuinely survives a
> process restart. The checkpoint-and-resume shape you see here is the same; only the
> storage backend changes.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

model = get_langgraph_model()

class State(TypedDict):
    dish: str
    approved: bool
    recipe: str

def propose_node(state):
    resp = model.invoke(
        "Pick ONE dinner dish from: butter chicken, southern fried burger, shawarma. "
        "Reply with just the dish name."
    )
    return {"dish": resp.content.strip()}

def approve_node(state):
    # ─── HITL GATE ─── graph PAUSES here on the first invoke. Whatever
    # value we pass to Command(resume=...) on the NEXT invoke is returned
    # by interrupt(). That value is the human's decision.
    decision = interrupt({"dish": state["dish"], "msg": "(approve/reject)"})
    return {"approved": decision == "approve"}

def recipe_node(state):
    if not state["approved"]:
        return {"recipe": f"REJECTED. No recipe for {state['dish']}."}
    resp = model.invoke(f"Give a short recipe for {state['dish']}.")
    return {"recipe": resp.content.strip()}

builder = StateGraph(State)
builder.add_node("propose", propose_node)
builder.add_node("approve", approve_node)
builder.add_node("recipe", recipe_node)
builder.add_edge(START, "propose")
builder.add_edge("propose", "approve")
builder.add_edge("approve", "recipe")
builder.add_edge("recipe", END)
graph = builder.compile(checkpointer=MemorySaver())

# Each conversation needs its own thread_id, otherwise resuming an old
# checkpoint would replay yesterday's decision. A timestamp keeps it fresh.
import time
config = {"configurable": {"thread_id": f"dinner-{int(time.time())}"}}

print("=" * 60)
print("LANGGRAPH — DINNER HITL (interrupt)")
print("=" * 60)

# Call 1 — runs propose_node, then PAUSES inside approve_node.
# interrupt() makes invoke() return early; state is checkpointed.
state = graph.invoke({}, config)
dish = state["dish"]
print(f"\nAgent proposes: {dish}")
print("(graph PAUSED at approval — state checkpointed)")

# ─── HITL GATE ─── genuine human prompt. The answer is injected into the
# frozen graph via Command(resume=...). Type 'reject' to watch it block.
decision = input(f"\nApprove {dish}? (approve/reject): ").strip().lower()
if decision not in ("approve", "reject"):
    decision = "reject"  # default-safe: ambiguous input → no recipe

# Call 2 — resume the frozen graph with the human's decision
state = graph.invoke(Command(resume=decision), config)
print(f"\nApproved: {state['approved']}")
print(f"--- RECIPE for {state['dish']} ---")
print(state["recipe"])

print("\nNote: MemorySaver here is in-process — fine for the demo. In production")
print("swap in SqliteSaver/PostgresSaver and the same code survives a kernel restart:")
print("the checkpointer + thread_id rebuilds state. That is the gold standard.")

LANGGRAPH — DINNER HITL (interrupt)

Agent proposes: Butter Chicken
(graph PAUSED at approval — state checkpointed)

Approved: True
--- RECIPE for Butter Chicken ---
Sure! Here's a simple recipe for Butter Chicken:

### Ingredients:
- 2 lbs (900g) chicken pieces (such as thighs and drumsticks)
- Salt, to taste
- Freshly ground black pepper, to taste

For the Marinade:
- 1/4 cup all-purpose flour
- 2 tablespoons vegetable oil
- 1 tablespoon minced garlic
- 1 teaspoon grated fresh ginger
- 30 ml (2 tablespoons) lemon juice
- 30 ml (2 tablespoons) freshly squeezed orange juice
- 1 large onion, finely chopped
- 4 cloves of garlic, minced
- 250ml (1 cup) chicken stock or water

For the Sauce:
- 250ml (1 cup) heavy cream
- 80g (3/4 cup) unsalted butter, cut into small pieces
- Salt and freshly ground black pepper to taste

### Instructions:

#### Step 1: Marinate the Chicken
1. Mix flour with salt and pepper in a large bowl.
2. Add chicken pieces and toss until fully coated.
3. Heat vegetabl

### Run 2 — reject at the gate

Same code as above; this time answer `reject` at the prompt to watch the interrupt resume with a denial.

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

model = get_langgraph_model()

class State(TypedDict):
    dish: str
    approved: bool
    recipe: str

def propose_node(state):
    resp = model.invoke(
        "Pick ONE dinner dish from: butter chicken, southern fried burger, shawarma. "
        "Reply with just the dish name."
    )
    return {"dish": resp.content.strip()}

def approve_node(state):
    # ─── HITL GATE ─── graph PAUSES here on the first invoke. Whatever
    # value we pass to Command(resume=...) on the NEXT invoke is returned
    # by interrupt(). That value is the human's decision.
    decision = interrupt({"dish": state["dish"], "msg": "(approve/reject)"})
    return {"approved": decision == "approve"}

def recipe_node(state):
    if not state["approved"]:
        return {"recipe": f"REJECTED. No recipe for {state['dish']}."}
    resp = model.invoke(f"Give a short recipe for {state['dish']}.")
    return {"recipe": resp.content.strip()}

builder = StateGraph(State)
builder.add_node("propose", propose_node)
builder.add_node("approve", approve_node)
builder.add_node("recipe", recipe_node)
builder.add_edge(START, "propose")
builder.add_edge("propose", "approve")
builder.add_edge("approve", "recipe")
builder.add_edge("recipe", END)
graph = builder.compile(checkpointer=MemorySaver())

# Each conversation needs its own thread_id, otherwise resuming an old
# checkpoint would replay yesterday's decision. A timestamp keeps it fresh.
import time
config = {"configurable": {"thread_id": f"dinner-{int(time.time())}"}}

print("=" * 60)
print("LANGGRAPH — DINNER HITL (interrupt)")
print("=" * 60)

# Call 1 — runs propose_node, then PAUSES inside approve_node.
# interrupt() makes invoke() return early; state is checkpointed.
state = graph.invoke({}, config)
dish = state["dish"]
print(f"\nAgent proposes: {dish}")
print("(graph PAUSED at approval — state checkpointed)")

# ─── HITL GATE ─── genuine human prompt. The answer is injected into the
# frozen graph via Command(resume=...). Type 'reject' to watch it block.
decision = input(f"\nApprove {dish}? (approve/reject): ").strip().lower()
if decision not in ("approve", "reject"):
    decision = "reject"  # default-safe: ambiguous input → no recipe

# Call 2 — resume the frozen graph with the human's decision
state = graph.invoke(Command(resume=decision), config)
print(f"\nApproved: {state['approved']}")
print(f"--- RECIPE for {state['dish']} ---")
print(state["recipe"])

print("\nNote: MemorySaver here is in-process — fine for the demo. In production")
print("swap in SqliteSaver/PostgresSaver and the same code survives a kernel restart:")
print("the checkpointer + thread_id rebuilds state. That is the gold standard.")

LANGGRAPH — DINNER HITL (interrupt)

Agent proposes: Butter Chicken
(graph PAUSED at approval — state checkpointed)

Approved: False
--- RECIPE for Butter Chicken ---
REJECTED. No recipe for Butter Chicken.

Note: between the two invokes the process could die and resume
later — the checkpointer + thread_id rebuilds state. That is the gold standard.


---
## Framework 3: CrewAI — `human_input=True` (with a workaround)

CrewAI's idiomatic HITL flag is `human_input=True` on a Task — when it works, the
task pauses and prompts the human for feedback via `input()`. Against the currently
installed `langchain`, that path crashes inside CrewAI with
`AttributeError: 'AgentExecutor' object has no attribute 'ask_for_human_input'`.

So we use the same pattern as the OpenAI SDK: a manual gate between two crew runs.
Task 1 proposes a dish, `input()` asks for approval, and Task 2 gives the recipe
only if the human says yes. Same propose → human gate → execute shape, no crash.

In [5]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

chef = Agent(
    role="Chef",
    goal="Pick one dish, then give its recipe once a human approves.",
    backstory="A friendly chef who picks from butter chicken, southern fried burger, or shawarma.",
    llm=llm,
    verbose=True,
)

print("=" * 60)
print("CREWAI — DINNER HITL")
print("=" * 60)

# NOTE: CrewAI's `human_input=True` on a Task is the idiomatic flag for
# HITL, but it is currently broken against the installed langchain —
# `AttributeError: 'AgentExecutor' object has no attribute 'ask_for_human_input'`.
# Until CrewAI/langchain align, we run a manual gate between two crew
# invocations: Task 1 proposes → input() → Task 2 recipes only if approved.
# Same propose → human gate → execute pattern as the other two frameworks.

# Task 1 — propose a dish
propose = Task(
    description=(
        "Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. "
        "Reply with just the dish name."
    ),
    expected_output="The dish name only.",
    agent=chef,
)

crew_propose = Crew(agents=[chef], tasks=[propose], process=Process.sequential, verbose=True)
proposed = await crew_propose.kickoff_async()
dish = str(proposed).strip()
print(f"\nAgent proposes: {dish}")

# ─── HITL GATE ─── genuine human prompt between two crew runs
decision = input(f"\nApprove {dish}? (yes/no): ").strip().lower()

# Task 2 — recipe for the approved dish, only if the human said yes
if decision == "yes":
    recipe = Task(
        description=f"Give a short recipe for {dish}.",
        expected_output="A short recipe.",
        agent=chef,
    )
    crew_recipe = Crew(agents=[chef], tasks=[recipe], process=Process.sequential, verbose=True)
    result = await crew_recipe.kickoff_async()
    print(f"\n--- RECIPE for {dish} ---")
    print(result)
else:
    print(f"\nREJECTED. No recipe for {dish}.")

print("\nNote: ideally `human_input=True` on one Task does this in a single")
print("crew, but that path is broken against current langchain. The two-crew")
print("manual gate is the reliable fallback — same HITL semantics, no crash.")

CREWAI — DINNER HITL


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 551c4d46-2af2-4162-9ba9-2e6b73a4ab06                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. Reply with just the dish  │
│  name.                                                                                                          │
│  ID: 80f42c9f-3c41-42f2-b743-678e46605e24                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chef                                                                                                    │
│                                                                                                                 │
│  Task: Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. Reply with just the dish  │
│  name.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chef                                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Certainly! Here's a simple and quick recipe for making delicious Shawarma:                                     │
│                                                                                                                 │
│  **Ingredients:**                                                                                               │
│  - 2 lbs of meat (chicken or beef are popular options)                                                          │
│  - Salt and pepper to taste                                                                                     │
│  - 1 cup harissa paste (or garlic, cumin, paprika, chili powder, and lemon juice mixed together)                │
│  - Oil for seasoning the chicken                                                                                │
│                                                                                                                 │
│  **Instructions:**                                                                                              │
│                                                                                                                 │
│  1. Preheat your grill at high heat.                                                                            │
│                                                                                                                 │
│  2. Lay whole pieces of meat seam-side up on a chopping board, place a knife between each piece and start to    │
│  score the meat with your blade about 1/4 inch deep - you should have an almost crisscross pattern.             │
│                                                                                                                 │
│  3. Season both sides of the chicken with salt and pepper; brush half your oil over one side of the shawarma    │
│  slices and keep some aside for basting (for grilling).                                                         │
│                                                                                                                 │
│  4. Put seasoned pieces of meat on a grill basket or skewer, arranging them shoulder to shoulder. Start         │
│  cooking immediately after applying oil.                                                                        │
│                                                                                                                 │
│  5. Cook for 10-12 minutes per side until charred and cooked thoroughly through; turn occasionally using two    │
│  brushes or paper towels dipped in water while the other oil is applied (this keeps the shawarma moist). Baste  │
│  with reserved oil halfway through cooking to ensure a crispy crust on both sides.                              │
│                                                                                                                 │
│  6. Once done, remove from heat, place on serving platter, drizzle with reserved oil and serve alongside your   │
│  favorite tahini sauce or yogurt dip for dipping. Enjoy!                                                        │
│                                                                                                                 │
│  Happy Cooking!                                                                                                 │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Give a short recipe for shawarma.                                                                        │
│  Agent: Chef                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 874b5823-2478-4f4c-a44e-6f6e2495fe7a                                                                       │
│  Final Output: Certainly! Here's a simple and quick recipe for making delicious Shawarma:                       │
│                                                                                                                 │
│  **Ingredients:**                                                                                               │
│  - 2 lbs of meat (chicken or beef are popular options)                                                          │
│  - Salt and pepper to taste                                                                                     │
│  - 1 cup harissa paste (or garlic, cumin, paprika, chili powder, and lemon juice mixed together)                │
│  - Oil for seasoning the chicken                                                                                │
│                                                                                                                 │
│  **Instructions:**                                                                                              │
│                                                                                                                 │
│  1. Preheat your grill at high heat.                                                                            │
│                                                                                                                 │
│  2. Lay whole pieces of meat seam-side up on a chopping board, place a knife between each piece and start to    │
│  score the meat with your blade about 1/4 inch deep - you should have an almost crisscross pattern.             │
│                                                                                                                 │
│  3. Season both sides of the chicken with salt and pepper; brush half your oil over one side of the shawarma    │
│  slices and keep some aside for basting (for grilling).                                                         │
│                                                                                                                 │
│  4. Put seasoned pieces of meat on a grill basket or skewer, arranging them shoulder to shoulder. Start         │
│  cooking immediately after applying oil.                                                                        │
│                                                                                                                 │
│  5. Cook for 10-12 minutes per side until charred and cooked thoroughly through; turn occasionally using two    │
│  brushes or paper towels dipped in water while the other oil is applied (this keeps the shawarma moist). Baste  │
│  with reserved oil halfway through cooking to ensure a crispy crust on both sides.                              │
│                                                                                                                 │
│  6. Once done, remove from heat, place on serving platter, drizzle with reserved oil and serve alongside your   │
│  favorite tahini sauce or yogurt dip for dipping. Enjoy!                                                        │
│                                                                                                                 │
│  Happy Cooking!                                                                                                 │
│                                                       


--- RECIPE for shawarma ---
Certainly! Here's a simple and quick recipe for making delicious Shawarma:

**Ingredients:**
- 2 lbs of meat (chicken or beef are popular options)
- Salt and pepper to taste
- 1 cup harissa paste (or garlic, cumin, paprika, chili powder, and lemon juice mixed together)
- Oil for seasoning the chicken

**Instructions:**

1. Preheat your grill at high heat.
   
2. Lay whole pieces of meat seam-side up on a chopping board, place a knife between each piece and start to score the meat with your blade about 1/4 inch deep - you should have an almost crisscross pattern.

3. Season both sides of the chicken with salt and pepper; brush half your oil over one side of the shawarma slices and keep some aside for basting (for grilling).

4. Put seasoned pieces of meat on a grill basket or skewer, arranging them shoulder to shoulder. Start cooking immediately after applying oil.

5. Cook for 10-12 minutes per side until charred and cooked thoroughly through; turn occasion

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Run 2 — reject at the gate

Same code as above; this time answer `no` at the prompt to watch the gate block the recipe.

In [6]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

chef = Agent(
    role="Chef",
    goal="Pick one dish, then give its recipe once a human approves.",
    backstory="A friendly chef who picks from butter chicken, southern fried burger, or shawarma.",
    llm=llm,
    verbose=True,
)

print("=" * 60)
print("CREWAI — DINNER HITL")
print("=" * 60)

# NOTE: CrewAI's `human_input=True` on a Task is the idiomatic flag for
# HITL, but it is currently broken against the installed langchain —
# `AttributeError: 'AgentExecutor' object has no attribute 'ask_for_human_input'`.
# Until CrewAI/langchain align, we run a manual gate between two crew
# invocations: Task 1 proposes → input() → Task 2 recipes only if approved.
# Same propose → human gate → execute pattern as the other two frameworks.

# Task 1 — propose a dish
propose = Task(
    description=(
        "Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. "
        "Reply with just the dish name."
    ),
    expected_output="The dish name only.",
    agent=chef,
)

crew_propose = Crew(agents=[chef], tasks=[propose], process=Process.sequential, verbose=True)
proposed = await crew_propose.kickoff_async()
dish = str(proposed).strip()
print(f"\nAgent proposes: {dish}")

# ─── HITL GATE ─── genuine human prompt between two crew runs
decision = input(f"\nApprove {dish}? (yes/no): ").strip().lower()

# Task 2 — recipe for the approved dish, only if the human said yes
if decision == "yes":
    recipe = Task(
        description=f"Give a short recipe for {dish}.",
        expected_output="A short recipe.",
        agent=chef,
    )
    crew_recipe = Crew(agents=[chef], tasks=[recipe], process=Process.sequential, verbose=True)
    result = await crew_recipe.kickoff_async()
    print(f"\n--- RECIPE for {dish} ---")
    print(result)
else:
    print(f"\nREJECTED. No recipe for {dish}.")

print("\nNote: ideally `human_input=True` on one Task does this in a single")
print("crew, but that path is broken against current langchain. The two-crew")
print("manual gate is the reliable fallback — same HITL semantics, no crash.")

CREWAI — DINNER HITL


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 87579be7-1331-4e76-806a-f150229fb3e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. Reply with just the dish  │
│  name.                                                                                                          │
│  ID: e2751d14-b31b-4392-853c-0d13e866ebdc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chef                                                                                                    │
│                                                                                                                 │
│  Task: Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. Reply with just the dish  │
│  name.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Propose ONE dinner dish from: butter chicken, southern fried burger, shawarma. Reply with just the dish  │
│  name.                                                                                                          │
│  Agent: Chef                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 87579be7-1331-4e76-806a-f150229fb3e0                                                                       │
│  Final Output: shawarma                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Agent proposes: shawarma


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


REJECTED. No recipe for shawarma.

Note: ideally `human_input=True` on one Task does this in a single
crew, but that path is broken against current langchain. The two-crew
manual gate is the reliable fallback — same HITL semantics, no crash.


---
## Comparison: Human-in-the-Loop

| Framework | Mechanism | Where the human gates | Resumable later? |
|---|---|---|---|
| OpenAI SDK | Gate between two `Runner.run()` calls | Your code, between steps | No (blocking) |
| LangGraph | `interrupt()` + `Command(resume=...)` | A checkpointed node | **Yes** |
| CrewAI | `human_input=True` *(broken)* → manual gate between two crews | Your code, between crews | No (blocking) |

**Key insight:** the pattern is always *propose → human gate → execute*. LangGraph is the
gold standard because the graph literally **freezes** — state is saved and you can resume
later from another session. The OpenAI SDK and CrewAI patterns are real (they genuinely
block on a human) but they hold a live call: if the process dies while waiting, the
approval is lost. CrewAI's `human_input=True` would be the simplest if it worked against
current langchain — until then, the manual gate carries the same semantics.

## Key Takeaway

Human-in-the-loop is essential for production agents. Not everything should be automatic.
- LangGraph: `interrupt()` — the graph freezes and resumes (most powerful)
- OpenAI SDK: a gate between two `Runner.run()` calls
- CrewAI: `human_input=True` is the simplest *when it works*; today a manual gate between
  two crews is the reliable path (see `AttributeError: ... ask_for_human_input`)

---